# 05. Impurities, LDOS and vortex-core views

Read the original LDOS code section by section, followed by the saved diagrams. Every displayed code block is copied from a supplied source file.

## 1. The file being explained

The code below is copied directly from your supplied [`Vortex Ldos copy.m`](https://github.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/blob/main/original_code/vortex_ldos_original.m). The stored file is unchanged. Each code block gives its original line numbers.

The explanations describe the assignments, indices, loops and plotting commands that are present. They do not add another solver, change the matrix convention, replace the LDOS expression, or supply missing plotting routines. The saved diagrams appear after the source walkthrough. A matching figure-producing run is not established simply by placing code beside an image.

The `.ipynb` is a reading notebook. The displayed code is Mathematica, and has not been executed in this editing pass.

## 2. Set the directory and calculation parameters

`SetDirectory` selects the directory used by the later relative imports and exports. Its path is the one saved in your file.

| Name | Value in the file | How it is used |
|---|---|---|
| `kernum` | 16 | Passed to `LaunchKernels`. |
| `Ns` | 10 | Used in both momentum loops and in the export divisor `Ns^2`. |
| `kkmx`, `kkmy` | `Ns*Nx`, `Ns*Ny` | Set the momentum increments and upper limits. |
| `wmax` | 0.10 | Sets the endpoints of the sampled energy interval. |
| `eta` | 0.005 | Appears as `I*eta` in the two spectral denominators. |
| `nw` | 400 | Divides the energy interval into 400 steps. |
| `ival` | Integers 1 through 200 | Selects the components used to calculate the site-resolved arrays. |

`Nx` and `Ny` are assigned later in the file. These earlier assignments therefore use whatever values or symbolic expressions those names have when evaluated.

**Original code: directory and sampling parameters** — `Vortex Ldos copy.m`, lines 3–12.

```mathematica
SetDirectory["/Users/aditya/Downloads/vortex 2"]
kernum = 16;

Ns = 10; (*supercell*)
kkmx = Ns*Nx;
kkmy = Ns*Ny;
wmax = 0.10;
eta = 0.005; (**)
nw = 400; 
ival = Table[i,{i,1,200}];
```

## 3. Define the hopping entries and site coordinates

`t[dx][dy]` assigns a number to each listed displacement. `l[dx][dy]` stores the four complex entries shown below. The script then sets `Nx=20`, `Ny=20` and increments `i` through a nested loop, storing `xpos[i]` and `ypos[i]`.

The inner loop runs through y first, so the index is `i=(x-1)*Ny+y`. After the loops, `NN=400`. In this ordering site 184 is `(10,4)`, and site 190 is `(10,10)`.

In this LDOS file, `t` is subsequently used to test the boundary connections. The normal Hamiltonian blocks themselves are imported from `.wdx` files; this section does not rebuild them from these amplitudes.

**Original code: hopping entries and coordinates** — `Vortex Ldos copy.m`, lines 14–38.

```mathematica
m = 0.50; 
lam = 1;(* next nearest neighbour hopping (diagonal)*)
t[0][0] = 3/2*m;
t[-1][0] = -m;
t[0][-1] = -m;
t[1][0] = -m;
t[0][1] = -m;
t[-1][-1] = m/4;
t[1][-1] = m/4;
t[-1][1] = m/4;
t[1][1] = m/4;
(*SOC spinorbit*)
l[-1][0] = lam(1-I);
l[0][-1] = lam(1-I);
l[1][0] = lam(1-I);
l[0][1] = lam(1-I);
Nx = 20;
Ny = 20;
i = 0;

For[ix = 1, ix <= Nx, ix++,
 For[iy = 1, iy <= Ny, iy++, i++;
  xpos[i] = ix;
  ypos[i] = iy;]]
NN = i;
```

## 4. Construct the x-boundary array

The nested `If` expressions check whether `t` has a numerical entry for each listed shifted displacement. A connection through an x-shifted copy receives `-1.` or `+1.` according to the branch shown. Cases that do not match a numerical entry do not become stored numerical entries in the final sparse array.

`pxij3[i][j] = pxijmn3[[j,i]]` transposes the temporary table. `Phasex` keeps its numerical entries. `Clear` removes the intermediate objects after they have been used.

**Original code: Phasex** — `Vortex Ldos copy.m`, lines 44–63.

```mathematica
Print["reading phasex3"]


pxijmn3 = Table[
If[NumericQ[t[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] - Ny)]], -1., 
If[NumericQ[t[xpos[i] - (xpos[j] - Nx)][ypos[i] - ypos[j]]], -1., 
If[NumericQ[t[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] + Ny)]], -1., 
If[NumericQ[t[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] - Ny)]],Plus[1.], 
If[NumericQ[t[xpos[i] - (xpos[j] + Nx)][ypos[i] - ypos[j]]], Plus[1.], 
If[NumericQ[t[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] + Ny)]], Plus[1.]]]]]]], 
     
{i,1,NN},{j,1,NN}]; 

For[i = 1, i <= NN, i++,
 For[j = 1, j <= NN, j++, 
   pxij3[i][j] = pxijmn3[[j,i]]]]
   
Clear[pxijmn3]
Phasex = SparseArray[{x_, y_} /; NumericQ[pxij3[x][y]] -> pxij3[x][y], {NN, NN}]
Clear[pxij3]
```

## 5. Construct the y-boundary array

This block follows the same pattern, now distinguishing connections through the shifted y copies. `pyijmn3` is the temporary table, `pyij3` holds its transposed entries, and `Phasey` is the sparse array retained for the phase factors.

**Original code: Phasey** — `Vortex Ldos copy.m`, lines 67–87.

```mathematica
Print["reading phasey3"]


pyijmn3 = Table[
If[NumericQ[t[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] - Ny)]], -1.,
If[NumericQ[t[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] + Ny)]], Plus[1.],
If[NumericQ[t[xpos[i] - xpos[j]][ypos[i] - (ypos[j] - Ny)]], -1., 
If[NumericQ[t[xpos[i] - xpos[j]][ypos[i] - (ypos[j] + Ny)]], Plus[1.],
If[NumericQ[t[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] - Ny)]], -1.,
If[NumericQ[t[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] + Ny)]], Plus[1.]]]]]]], 
  
   {i,1,NN},{j,1,NN}]; 
    
For[i = 1, i <= NN, i++,
 For[j = 1, j <= NN, j++, 
 
   pyij3[i][j] = pyijmn3[[j,i]]]]
   
Clear[pyijmn3]
Phasey = SparseArray[{x_, y_} /; NumericQ[pyij3[x][y]] -> pyij3[x][y], {NN, NN}]
Clear[pyij3]
```

## 6. Import the saved matrices

The five `Import` calls load `Huu.wdx`, `Hud.wdx`, `Hdu.wdx`, `Hdd.wdx` and `SCs.wdx` from the selected directory. The filenames alone do not specify which impurity or vortex calculation created their contents.

The following `empty` assignment is reproduced exactly. Its condition applies `NumericQ` to `2==2`, which evaluates to the Boolean `True`; that Boolean is not a number. Thus the rule does not insert the value 2 at matrix positions, and the unspecified sparse entries are zero.

**Original code: matrix imports and empty block** — `Vortex Ldos copy.m`, lines 90–102.

```mathematica
Print["Reading H matrix now"]
Huu = Import["Huu.wdx"]; 
Hud = Import["Hud.wdx"]; 
Hdu = Import["Hdu.wdx"]; 
Hdd = Import["Hdd.wdx"]; 
SCs = Import["SCs.wdx"];
Print["Reading H matrix done"]


(*Dimensions[Phasex1]*)


empty=SparseArray[{x_,y_}/;NumericQ[2==2]->2,{NN,NN}];
```

## 7. Assemble H as written in the file

The first four assignments multiply the imported normal blocks element by element by the exponential boundary factor. `Re[kx]` and `Re[ky]` take real parts. `hh1` puts those four blocks into a larger matrix with `ArrayFlatten`.

`SCs1` takes rows and columns `1` through `NN`; `SCs2` takes rows and columns `NN+1` through `2 NN`. These are the two diagonal portions of imported `SCs`. Both `SCs1c` and `SCs2c` are then calculated from **`SCs1`**, exactly as the file states. Although `SCs2` is assigned, it is not used in those two expressions.

`hh2` uses the displayed minus signs and complex conjugates. `huv` uses `SCs1c` and `SCs2c` in its off-diagonal blocks. Finally `H` combines `hh1`, `huv`, `Conjugate[huv]` and `hh2`.

`H == ConjugateTranspose[H];` is an equality expression with its output suppressed by the semicolon. It does not stop the calculation if equality fails. This walkthrough leaves all these expressions unchanged; it does not substitute another block convention.

**Original code: phase-dependent matrix assembly** — `Vortex Ldos copy.m`, lines 106–122.

```mathematica
Clear[kx, ky]
Huuc = Huu*Exp[I*Re[kx]*Nx*Phasex + I*Re[ky]*Ny*Phasey]; 
Hudc = Hud*Exp[I*Re[kx]*Nx*Phasex + I*Re[ky]*Ny*Phasey]; 
Hduc = Hdu*Exp[I*Re[kx]*Nx*Phasex + I*Re[ky]*Ny*Phasey]; 
Hddc = Hdd*Exp[I*Re[kx]*Nx*Phasex + I*Re[ky]*Ny*Phasey]; 

hh1=ArrayFlatten[{{Huuc,Hudc},{Hduc,Hddc}}];
SCs1=Take[SCs,{1,NN},{1,NN}];
SCs2=Take[SCs,{NN+1,2NN},{NN+1,2NN}];
SCs1c=SCs1*Exp[I*Re[kx]*Nx*Phasex + I*Re[ky]*Ny*Phasey];
SCs2c=SCs1*Exp[I*Re[kx]*Nx*Phasex + I*Re[ky]*Ny*Phasey];
hh2 = ArrayFlatten[{{-Conjugate[Huuc],Conjugate[Hudc]}, {Conjugate[Hduc], -Conjugate[Hddc]}}];

huv=ArrayFlatten[{{empty,SCs1c},{SCs2c,empty}}];
H=ArrayFlatten[{{hh1,huv},{Conjugate[huv],hh2}}];

H == ConjugateTranspose[H];
```

## 8. Diagonalize at each momentum and form the LDOS array

`CloseKernels[]` closes the parallel kernels, and `LaunchKernels[kernum]` requests the specified number. `ParallelTable` runs the body for the displayed `kx` and `ky` values. The two `For` lines inside comments are inactive.

Inside the table:

1. `ES = Eigensystem[H]` obtains the eigenvalues and eigenvectors.
2. `elist` copies all `4*NN` eigenvalues from `ES[[1]]`.
3. `ulist` takes squared magnitudes of the eigenvector components indexed by `ival`.
4. `vlist` takes squared magnitudes of the components indexed by `ival+2*NN`.
5. For each `w`, the inner sum adds the two displayed denominator terms for every eigenstate, then `Im` takes the imaginary part.

The sampled energy is `-wmax+2*wmax*w/nw`. With `w=0,...,400`, the table contains 401 energies from -0.10 to +0.10. Each energy row contains a value for each of the 200 selected indices.

`Flatten[...,1]` combines the two momentum-table levels into a list of LDOS arrays. The outer `Total` sums those arrays. The result is stored in `ldos`; the normalization factor is applied later when writing the data.

**Original code: original parallel spectral sum** — `Vortex Ldos copy.m`, lines 131–148.

```mathematica
Print["starting ldos calc"];
 CloseKernels[];
 LaunchKernels[kernum];
ldos=Total[Flatten[ParallelTable[
(*For[kx = 0, kx <= (Ns - 1.)*2*(Pi/kkmx), kx += 2*(Pi/kkmx),
 For[ky = 0, ky <= (Ns - 1.)*2*(Pi/kkmy), ky += 2*(Pi/kkmy),*)
  ES = Eigensystem[H];
   elist = Table[ES[[1,l]], {l, 1, 4*NN}];
 
   ulist=Table[Abs[ES[[2,l,ival]]]^2,{l,1,4*NN}(*,{i,1,(nsite-1)*Nband+Nband}*)];
vlist=Table[Abs[ES[[2,l,ival+2*NN]]]^2,{l,1,4*NN}(*,{i,1,(nsite-1)*Nband+Nband}*)];


ldossc=Table[Im[Total[Table[(ulist[[l,;;]]) (1/(-wmax+wmax*2*w/nw-elist[[l]]+I*eta))+
(vlist[[l,;;]]) (1/(-wmax+2*wmax*w/nw+elist[[l]]+I*eta)),{l,1,4*NN}]]],{w,0,nw}]; ldossc,
{kx,0,(Ns - 1.)*2*(Pi/kkmx),2*(Pi/kkmx)},{ky,0,(Ns - 1.)*2*(Pi/kkmy),2*(Pi/kkmy)}],1]];
      (*ldos = ldos + ldossc;*) 
      Print["done"];
```

## 9. Write the result to a file

The commented `Export` alternative is inactive. The active code constructs `fl`, deletes the named existing output, opens an append stream, and writes one row for every energy index `n` and selected-site index `i`.

Each row contains the energy and `(-1/Pi)*ldos[[n,i]]/Ns^2`. The site index is not written as a separate column. Because the site loop is inside the energy loop, rows for all selected sites at one energy are written before moving to the next energy.

The filename expression uses the Greek symbol `\[Eta]`, while the broadening used above is the separately named `eta`. The file is explained as written; neither symbol is replaced here.

**Original code: original export** — `Vortex Ldos copy.m`, lines 154–166.

```mathematica
  (*Export["ldos_sup.dat",(* ldos*)Table[{-wmax + wmax*2*((n-1)/nw),(-Pi^(-1))*ldos[[n,i]]/Ns^2}, {n, 1, nw+1}, {i, 1, Length[ival]}]]

Print["done"];*)
fl = StringJoin["wmax",ToString[wmax],"_Ny",ToString[Ny],"_Nx",ToString[Nx],
"_eta",ToString[\[Eta]],"_Ns",ToString[Ns]];
DeleteFile[StringJoin[fl,"_ldos_sup_cdw.dat"]];
str1 = OpenAppend[StringJoin[fl,"_ldos_sup_cdw.dat"], FormatType -> StandardForm];
For[n=1,n<=nw+1,n++,
For[i=1,i<=Length[ival],i++,
Write[str1,-wmax + wmax*2*((n-1)/nw),"  ", (-1/Pi)*ldos[[n,i]]/Ns^2];
]
];
Close[str1];
```

## 10. Read and plot the exported table

`Import[...,"Table"]` loads the specific filename shown. `ListPlot[Lds, Joined->True]` joins the imported points. This command does not separate the interleaved rows into individual site curves.

`FrameLabel` supplies the displayed labels. `GridLines` places the two red reference lines at -0.04 and +0.04, and `Epilog` places the two text annotations. These values are explicitly supplied plotting settings; this command does not calculate them from the eigenvalues.

**Original code: import and energy-labelled plot** — `Vortex Ldos copy.m`, lines 174–176.

```mathematica
Lds= Import["wmax0.1_Ny20_Nx20_eta\[Eta]_Ns10_ldos_sup_cdw.dat","Table"];
ListPlot[Lds, Joined->True, PlotRange->All,ImageSize->Large, Frame->True, FrameLabel->
{Energy[eV],LDOS},GridLines -> {{{0.04, Red}, {-0.04, Red}},None }, GridLinesStyle-> Dotted,  Epilog -> {Text["(\[CapitalDelta]=0.04)", {0.05, 8}],Text["(\[CapitalDelta]=-0.04)", {-0.05, 8}]}]
```

## 11. Read the final site-selection plot

The active expression selects `ldos[[n,184]]` while `n` runs from 1 to `nw`. Since `ival` is the sequence 1 through 200, column 184 corresponds to selected site 184 here. The alternative column 1 expression is commented out.

This `ListPlot` receives values without explicit x coordinates. Its horizontal coordinate is therefore the list index. The loop varies the energy-row index while holding the selected site fixed; it is not a scan through lattice sites. It uses 400 rows, omitting the last of the 401 energy rows calculated above, and applies `-1/Pi` without the `/Ns^2` factor used by the export.

This explains this particular command. It does not establish that it generated the saved three-colour impurity diagrams.

**Original code: final active plot** — `Vortex Ldos copy.m`, lines 195–195.

```mathematica
ListPlot[ {Table[(-1/Pi)*ldos[[n,184]],{n,1,nw}](*,Table[(-1/Pi)*ldos[[n,1]],{n,1,nw}]*)}, Joined->True]
```

## 12. Commented plotting alternatives

The file also contains the commented `ListDensityPlot` expression below. Because it is inside `(* ... *)`, it does not execute. Its frame labels are `Position` and `Energy`. It is not an active two-dimensional or three-dimensional vortex-core plotting routine.

**Original code: inactive density-plot alternative** — `Vortex Ldos copy.m`, lines 188–192.

```mathematica
(*LDos1 = Import["wmax0.15_Ny20_Nx20_eta\[Eta]_Ns20_ldos_sup_cdw.dat","Table"];
En = Import["Energy.dat","Table"];
ListDensityPlot[LDos1,  ColorFunction -> "TemperatureMap", 
  PlotRange -> Full, PlotLegends -> Automatic, 
  FrameLabel -> {"Position", "Energy"}, ImageSize -> Large]*)
```

## 13. What the original files contain about impurities

In `BdG topological.nb`, `Vimp=0` is assigned, but the displayed `+imp` contribution is commented out. The displayed active spin-block assignments therefore do not run a non-magnetic or magnetic impurity sweep. The LDOS file loads saved matrices rather than adding those impurity terms itself. The missing sweep code is not reconstructed here.

**Original impurity parameter** — `BdG topological.nb`, input 1.

```mathematica
Vimp = 0;

```

**Original spin-block assignments** — `BdG topological.nb`, input 5.

```mathematica
  Huu = H1-Chempot1[mu] (*+imp*)(**+Hartree*);
  Hdd = H1-Chempot1[mu];
Hud = soc;
Hdu = Conjugate[soc];
hh1 = ArrayFlatten[{{Huu,Hud}, {Hdu, Hdd}}];
hh2 = ArrayFlatten[{{-Conjugate[Huu],Conjugate[Hud]}, {Conjugate[Hdu], -Conjugate[Hdd]}}];

```

## 14. The vortex update present in added vortex.nb

This is the actual vortex-related block in `added vortex.nb`. It sets `vsite=378`, `l=0.3` and `zetanot=l/Pi`. It calculates `phi` using the displayed absolute coordinate differences, forms the eigenvector-product sum, multiplies by `vs`, and multiplies by the `Tanh` and `Exp[I*phi]` factors to form `stcalc`.

`xpos` and `ypos` refer to the coordinate arrays built earlier in that file. Its initial lattice assignments are `Nx=28` and `Ny=28`; these settings must not be silently replaced by the settings printed on a different saved image.

The `Tanh` argument uses the distance from the selected site. `Exp[I*phi]` multiplies the result by a complex phase. The expressions below are left exactly as supplied, including the absolute-value angle formula. This is a field-update block; it contains no heatmap or surface-plot command. The provided files do not establish the exact commands that produced the saved two-dimensional and three-dimensional core views.

**Original vortex contribution to stcalc** — `added vortex.nb`, input 5.

```mathematica
vsite = 378;
  l=0.3;(*penetration depth*)
  zetanot=l/Pi;(*coherence length*)
    stcalc=Table[
  If[xpos[i]==xpos[vsite],
  phi=Pi/2,
  phi=ArcTan[(Abs[ypos[i]-ypos[vsite]])/(Abs[xpos[i]-xpos[vsite]])];
  ];
  vs*((Inner[Times,ES[[2]][[;;,i]],Conjugate[ES[[2]][[;;,j+
  3NN]]],List])+(Inner[Times,ES[[2]][[;;,i+NN]],Conjugate[ES[[2]][[;;,i+
  2NN]]],List])).fermi*
  Tanh[Sqrt[(xpos[i]-xpos[vsite])^2+(ypos[i]-ypos[vsite])^2]/zetanot]*
  Exp[I*phi]
,{i,1,NN},{j,js[[i]]}];

```

## 15. The original spectrum plotting command

This command plots `Sort[ES[[1]]]`, the sorted eigenvalues. The title, range and reference lines are specified explicitly. It is copied from the original file; those settings do not demonstrate that this command produced every archived spectrum shown below.

**Original spectrum plot** — `BdG topological.nb`, input 7.

```mathematica
ListPlot[Sort[ES[[1]]],PlotRange\[Rule]{-0.6,0.6} ,Frame\[Rule]True,PlotLabel\[Rule]"\<\!\(\*SubscriptBox[\(\[CapitalDelta]\), \(o\)]\)=\[PlusMinus]0.7, M=0, m=0.5, \[Lambda]=1, <n>=1, \!\(\*SubscriptBox[\(\[CapitalDelta]\), \(con\)]\)=0.064, \!\(\*SubscriptBox[\(\[Mu]\), \(con\)]\)=-0.46 \>",FrameLabel\[Rule]{n,Energy},LabelStyle\[Rule]"\<DisplayFormula\>",GridLines\[Rule]{None,{-0.05,0.05}},GridLinesStyle\[Rule]Directive[Green, Dashed], Frame\[Rule]True, FrameStyle\[Rule]Gray,ImageSize\[Rule]Large]
```

## Saved energy-axis LDOS diagrams

The following are archived outputs. Their original energy labels and scales are retained; the exact generating run for each image has not been verified from the supplied files.

### LDOS with a narrow central peak

![LDOS with a narrow central peak](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_central_peak.png)

The blue curve has a large peak close to zero energy, with many smaller oscillations on either side. The curve remains appreciable through the central region. The small oscillations are features of the saved curve; the image alone does not establish their numerical origin.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_central_peak.png)

### LDOS suppressed between two sharp peaks

![LDOS suppressed between two sharp peaks](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_gap.png)

Red dotted reference lines lie at approximately -0.06 and +0.06 meV. Between them, the LDOS is close to zero; near either line it rises sharply. The two sides are not identical in height and background. These plotted values should be read directly rather than assumed symmetric.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_gap.png)

### Two peaks inside the marked energy interval

![Two peaks inside the marked energy interval](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_two_inner_peaks.png)

The red dotted lines mark approximately ±0.05 meV. Two broad maxima appear inside that interval, with a smaller but nonzero LDOS near zero energy. A central dip and two finite-energy peaks are different from a zero-energy peak.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_two_inner_peaks.png)

## Saved pairing line profiles

The three panels below plot pairing amplitude against position. Their horizontal coordinate is spatial, unlike the energy axes immediately above.

### Three spatial profiles with different local distortion

![Three spatial profiles with different local distortion](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_profiles_impurity.png)

All three orange curves approach zero near the central coordinate. From left to right, the neighbouring region becomes more uneven, while values far from the centre stay around the same general level. The panel scales are slightly different: the rightmost vertical scale reaches about 0.08, whereas the first two reach about 0.07. The colours identify the plotted line, not three distinct observables.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_profiles_impurity.png)

## Saved impurity LDOS curves

The next six saved images label the horizontal axis **Number of Sites**. That is the archived label. The exact generating command for these three-colour curves is not supplied. The active final command above selects column 184 and contains one active curve; it is not evidence for the site identities of the green, blue and orange curves in these images.

The panels are grouped by their stored scalar/nonmagnetic or magnetic case. The descriptions below state what is visible; they do not assume that changes in peak height prove protection or identify a particular bound state.

### Scalar case: green curve above the other two

![Scalar case: green curve above the other two](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_weak.png)

The green trace lies above the blue and orange traces through much of the index range. All three curves show a deep minimum around index 100, with larger values on both sides. The green minimum remains slightly higher than the others.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_weak.png)

### Scalar case: a reduced green trace

![Scalar case: a reduced green trace](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_middle.png)

The green curve is much smaller than the other two over most indices. The blue and orange curves retain a central depression and prominent shoulders. The orange trace is higher around several shoulder peaks, so colour order is not fixed across all cases.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_middle.png)

### Scalar case: green trace close to the baseline

![Scalar case: green trace close to the baseline](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_strong.png)

The green trace is nearly flat at zero. Blue and orange retain a substantial signal away from the central dip, with the blue trace generally larger. The result is strongly position dependent; it is not a uniform rescaling of every curve.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_scalar_strong.png)

### Magnetic case: three resolved LDOS traces

![Magnetic case: three resolved LDOS traces](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_weak.png)

All three traces have a central trough. The blue curve has the largest shoulder peaks, while green and orange cross in parts of the outer region. Compare the traces at the same horizontal index to assess their relative values.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_weak.png)

### Magnetic case: suppression of the green trace

![Magnetic case: suppression of the green trace](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_middle.png)

The green curve is low across the plotted range. Blue and orange show pronounced shoulders surrounding the central trough. The blue shoulders are taller, but the two outer traces remain similar in some regions.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_middle.png)

### Magnetic case: green trace near zero

![Magnetic case: green trace near zero](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_strong.png)

The green result stays close to the baseline. The remaining two traces still have a broad central minimum, with blue generally above orange. The vertical scale differs from other panels, so identical apparent heights on screen need not represent identical values.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/ldos_magnetic_strong.png)

## Saved impurity spectra

In each spectrum below, horizontal n is the index in a sorted energy list. Blue dots are eigenvalues. Green dashed lines mark the reference values drawn in each image. The original spectrum command above also specifies reference lines, but its settings are not verified as the settings for every image.

### Scalar spectrum: levels inside the reference interval

![Scalar spectrum: levels inside the reference interval](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_scalar_a.png)

Several blue dots lie between the green lines around zero. More distant levels occur above and below that interval. This plot shows energies, not the locations of their wavefunctions.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_scalar_a.png)

### Magnetic spectrum: small positive and negative levels

![Magnetic spectrum: small positive and negative levels](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_magnetic_a.png)

Blue levels occur on both sides of zero, including inside the marked interval. Closely spaced points can overlap at this image resolution, so an exact degeneracy count should not be inferred from visible dots alone.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_magnetic_a.png)

### Scalar spectrum with a near-zero level

![Scalar spectrum with a near-zero level](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_scalar_b.png)

A dot lies close to zero and other levels lie near the reference boundaries. The title records the parameters attached to this saved plot. A near-zero energy alone does not establish the state’s spatial distribution or physical identity.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_scalar_b.png)

### Magnetic spectrum with a near-zero level

![Magnetic spectrum with a near-zero level](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_magnetic_b.png)

The same types of features are visible: a central near-zero level, nearby levels and an outer sequence of states. The broad resemblance to another spectrum does not establish equality of the underlying matrices or wavefunctions.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_magnetic_b.png)

## Saved spatial maps and surfaces

Colour maps place one cell at each lattice coordinate. The surface diagrams also show a vertical numerical axis. Several of these surfaces have **zero at the top of the vertical axis and larger positive values lower down**. An apparent upward spike can therefore correspond to a smaller value; read the ticks before describing it as an enhancement.

### Scalar case: a distinct central cell

![Scalar case: a distinct central cell](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_scalar_weak.png)

A pale cell near (10,10) stands out from the surrounding orange and yellow cells. The coordinates span a 20 × 20 grid, with y increasing downward. There is no colour bar, so exact amplitudes cannot be recovered from the colours.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_scalar_weak.png)

### Scalar surface with an inverted vertical scale

![Scalar surface with an inverted vertical scale](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_scalar_a.png)

The surface is uneven and a sharp central feature rises toward the tick marked zero. Larger positive tick values are lower in the drawing. Thus its visual upward direction must not be interpreted as an increase of the plotted quantity.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_scalar_a.png)

### Scalar case: irregular surrounding cells

![Scalar case: irregular surrounding cells](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_scalar_strong.png)

The central region is pale, while surrounding cells fluctuate more strongly between yellow and orange. The map displays spatial structure but supplies no numerical colour scale. It does not show the phase of a complex field.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_scalar_strong.png)

### Scalar surface with a central feature and rough surroundings

![Scalar surface with a central feature and rough surroundings](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_scalar_b.png)

The vertical ticks run from about 0 near the top toward 0.08 below. The central feature points toward smaller values, while the surrounding surface is uneven. Its vertical range differs from the preceding scalar surface.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_scalar_b.png)

### Magnetic surface with a pronounced central feature

![Magnetic surface with a pronounced central feature](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_magnetic.png)

The vertical scale again increases downward, from 0 toward about 0.06. The central feature reaches toward the smaller-value end of that scale. Away from the centre, the mesh has substantial site-to-site variation.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_surface_magnetic.png)

### Magnetic case: patchy spatial pattern

![Magnetic case: patchy spatial pattern](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_magnetic.png)

Orange and yellow cells alternate across much of the grid, and a pale central cell remains visible. The plot gives spatial coordinates but no numerical colour key. It supports a description of the pattern, not a precise amplitude comparison with another colour map.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/pairing_map_magnetic.png)